In [1]:
%pip install pydeseq2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 113.9 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scispacy 0.6.2 requires numpy<2.0, but you have numpy 2.5.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install archs4py

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 21.4 MB/s  0:00:00 18.5 MB/s eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 113.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 161.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 153.2 MB/s  0:00:00
  Created wheel for archs4py: filename=archs4py-1.0.4-py3-none-any.whl size=18502 sha256=6288b611306e633daf1ba71e8d5d44c9d3d9f62f9c4ba04670255caa4e94ddb1
  Stored in directory: /home/vik/.cache/pip/wheels/86/61/c7/0e4580ce6350339f0ac5915c8e1b3c2b9481f98142612be521
  C

In [4]:
%wget https://s3.k8s.maayanlab.cloud/archs4/files/human_gene_v2.latest.h5

UsageError: Line magic function `%wget` not found.


In [3]:
#imports
from pathlib import Path
import pandas as pd
import archs4py as a4
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
#load ground control and space flight from specified cohort (copy from cluster_metadata)

def get_counts_metadata(cohort_name,clist): #returns counts and metadata dataframes
    human_archs4_path = '/media/volume/H5-Files/archs4/human_gene_v2.latest.h5'
    mouse_archs4_path = '/home/vik/bridge-rna/archs4h5/mouse_gene_v2.5.h5'
    folder = Path.cwd().parent / "archs4data"/ "archs4metadata_cohort_noncbi"
    full_name = folder / f"{cohort_name}_hits.csv"
    ch_df = pd.read_csv(full_name)
    
    #select GSM,spaceflight
    metadata = ch_df[['gsm','spaceflight']]
    metadata = metadata.groupby('gsm').filter(lambda x : len(x)<2) #drop samples that appear in ground and space
    metadata = metadata.set_index("gsm")
    metadata.index.name = None
    metadata = metadata[metadata['spaceflight'].isin(clist)]
    metadata = metadata.sort_index()
    

    #get counts
    gsms = metadata.index.tolist()
    #print(gsms)
    #gsm may be in human or mouse. but would they ever be mixed human and mouse?
    #for now just try mouse and if it fails then go human
    
    sample_counts = a4.data.samples(mouse_archs4_path,gsms)
    if sample_counts.empty:
        print("Human isn't downloaded yet")
        raise ValueError("Human isn't downloaded yet")
        sample_counts = a4.data.samples(human_archs4_path,gsms)
    sample_counts = sample_counts.groupby(sample_counts.index).agg(func='sum') #collapses all the dupes
    sample_counts = sample_counts.T
    sample_counts = sample_counts.sort_index()
    
    
    
    return sample_counts,metadata
        
    
    

In [85]:

cohort = 'OSD-667'
counts_df,metadata_df = get_counts_metadata(cohort,['Ground Control','Space Flight'])


100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 32.38it/s]


In [86]:
metadata_df["spaceflight"] = pd.Categorical(
    metadata_df["spaceflight"],
    categories=["Ground Control", "Space Flight"],
    ordered=True
)

In [87]:
counts_df

,0610005C13Rik,0610006L08Rik,0610009B22Rik,0610009E02Rik,0610009L18Rik,0610010K14Rik,0610012D04Rik,0610012G03Rik,0610025J13Rik,0610030E20Rik,...,n-R5s58,n-R5s60,n-R5s64,n-R5s65,n-R5s7,n-R5s71,n-R5s86,n-R5s88,n-R5s92,n-R5s93
GSM2593658,154,0,120,24,41,166,6,453,2,491,...,0,0,0,0,0,0,0,0,0,0
GSM2593665,93,0,127,30,59,184,8,549,0,512,...,0,0,0,0,0,0,0,0,0,0
GSM2593668,82,0,129,27,54,210,5,568,0,540,...,0,0,0,0,0,0,0,0,0,0
GSM2950654,98,0,95,14,22,138,7,376,0,461,...,0,0,0,0,0,0,0,0,0,0
GSM2950660,176,0,111,29,37,153,7,382,2,464,...,0,0,0,0,0,0,0,0,0,0
GSM2950672,194,0,138,26,31,169,6,417,1,482,...,0,0,0,0,0,0,0,0,0,0


In [88]:
counts_df.columns[counts_df.columns.duplicated()]

Index([], dtype='str')

In [89]:
metadata_df.index.tolist()

['GSM2593658',
 'GSM2593665',
 'GSM2593668',
 'GSM2950654',
 'GSM2950660',
 'GSM2950672']

In [90]:
metadata_df

,spaceflight
GSM2593658,Ground Control
GSM2593665,Space Flight
GSM2593668,Space Flight
GSM2950654,Ground Control
GSM2950660,Ground Control
GSM2950672,Space Flight


In [91]:
#load them into the deseq format
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata_df,
    design="spaceflight",  # compare samples based on the "condition"
    # column ("B" vs "A")
    refit_cooks=True,
    inference=inference,
)

In [92]:
'''
Size factors: Scaling factors, for normalization
Genewise dispersions: variability in each gene's counts accross replicates
Replicate: samples with the same condition (i.e Ground)
Trend Coeff: Dispersion-mean trend (smooth curve). The coeffs are the coeffs for the parametric model
*for DE Analysis, it is not necessary to use gene lengths to estimate the scaling factors (it is even not recommended) (i.e typical normalization methods)
Deseq uses a different normalization method
MAP dispersion: applies shrinkage; uses Bayes empirical shrinkage
LFC: Log fold change

'''

"\nSize factors: Scaling factors, for normalization\nGenewise dispersions: variability in each gene's counts accross replicates\nReplicate: samples with the same condition (i.e Ground)\nTrend Coeff: Dispersion-mean trend (smooth curve). The coeffs are the coeffs for the parametric model\n*for DE Analysis, it is not necessary to use gene lengths to estimate the scaling factors (it is even not recommended) (i.e typical normalization methods)\nDeseq uses a different normalization method\nMAP dispersion: applies shrinkage; uses Bayes empirical shrinkage\nLFC: Log fold change\n\n"

In [93]:
dds.fit_size_factors() 
dds.obs["size_factors"]

Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.03 seconds.



GSM2593658    1.007989
GSM2593665    1.205046
GSM2593668    1.100642
GSM2950654    0.854391
GSM2950660    0.947468
GSM2950672    0.951356
Name: size_factors, dtype: float64

In [ ]:
dds.fit_genewise_dispersions()
dds.var["genewise_dispersions"]

In [ ]:
dds.fit_dispersion_trend()
dds.uns["trend_coeffs"]
dds.var["fitted_dispersions"]

In [ ]:
dds.fit_dispersion_prior()
print(
    f"logres_prior={dds.uns['_squared_logres']}, sigma_prior={dds.uns['prior_disp_var']}"
)

In [ ]:
dds.fit_MAP_dispersions()
dds.var["MAP_dispersions"]
dds.var["dispersions"]

In [ ]:
dds.fit_LFC()
dds.varm["LFC"]

In [ ]:
dds.calculate_cooks()
if dds.refit_cooks:
    # Replace outlier counts
    dds.refit()

In [ ]:
ds = DeseqStats(
    dds,
    contrast=["spaceflight", "Space Flight", "Ground Control"],
    alpha=0.05,
    cooks_filter=True,
    independent_filter=True,
)

In [ ]:
ds.run_wald_test()
ds.p_values

In [ ]:
ds.summary()

In [ ]:
#visualization 
df = ds.results_df

In [ ]:
df.columns

In [ ]:
#save summary to folder

df.to_csv(f"deseq_summaries/{cohort}",index=True)

In [ ]:
#test load
read_cohort = 'OSD-667'
loaded_df = pd.read_csv(f'deseq_summaries/{read_cohort}',index_col=0)

In [ ]:
loaded_df

In [ ]:
#pydeseq2 imports are not working, this is a direct copy paste of their function
from typing import Literal
def make_MA_plot(
    results_df: pd.DataFrame,
    padj_thresh: float = 0.05,
    log: bool = True,
    save_path: str | None = None,
    lfc_null: float = 0,
    alt_hypothesis: Literal["greaterAbs", "lessAbs", "greater", "less"] | None = None,
    **kwargs,
) -> None:
    """
    Create an log ratio (M)-average (A) plot using matplotlib.

    Useful for looking at log fold-change versus mean expression between two groups/samples/etc.
    Uses matplotlib to emulate the ``make_MA()`` function in DESeq2 in R.

    Parameters
    ----------
    results_df
        Resultant dataframe after running DeseqStats() and .summary().
    padj_thresh
        P-value threshold to subset scatterplot colors on.
    log
        Whether or not to log scale features and targets axes (``default=True``).
    save_path
        The path where to save the plot.
        If left None, the plot won't be saved (``default=None``).
    lfc_null
        The (log2) log fold change under the null hypothesis. (default: ``0``).
    alt_hypothesis
        The alternative hypothesis for computing wald p-values. (default: ``None``).
    **kwargs
        Matplotlib keyword arguments for the scatter plot.
    """
    colors = results_df["padj"].apply(lambda x: "darkred" if x < padj_thresh else "gray")

    fig, ax = plt.subplots(dpi=600)

    # Set default alpha and s parameters, if not already specified
    kwargs.setdefault("alpha", 0.5)
    kwargs.setdefault("s", 0.2)

    plt.scatter(
        x=results_df["baseMean"],
        y=results_df["log2FoldChange"],
        c=colors,
        **kwargs,
    )

    ax.set_adjustable("datalim")

    if log is True:
        plt.xscale("log")

    plt.xlabel("mean of normalized counts")
    plt.ylabel("log2 fold change")
    plt.title(f"MA Plot: {read_cohort} Ground Control ARCHS4 Hits vs Spaceflight ARCHS4 Hits")

    plt.axhline(lfc_null, color="red", alpha=0.5, linestyle="--", zorder=3)
    if alt_hypothesis and alt_hypothesis in ["greaterAbs", "lessAbs"]:
        plt.axhline(-lfc_null, color="red", alpha=0.5, linestyle="--", zorder=3)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight")

In [ ]:
make_MA_plot(loaded_df)


In [ ]:
##gsea?###

